In [ ]:
%%configure -f
{
    "defaultLakehouse": { "name": "SilverLakehouse" }
}

# Silver - Grad Exams (Cosmos)  ·  Issue #29
Reads the **Bronze Graduate Exams mirror** (Cosmos `gradExams` mirrored into OneLake) and
writes flattened **Silver** grad-exam facts into `SilverLakehouse` under the **`dbo`** schema.
**What it does**
1. Attaches `SilverLakehouse` as the default lakehouse (by name - no tenant GUIDs committed).
2. Resolves the Graduate Exams mirror item at runtime and reads `university/gradExams` by item id.
3. Flattens the nested `undergradContext` (robust to struct-or-JSON-string storage) and
   explodes `relevantCourses[]` into a long table.
4. Writes managed Delta tables `dbo.grad_exam_result` and `dbo.grad_exam_relevant_course`.
**Silver output** (answers demo question 6, the SQL <-> Cosmos join on `student_id`).

In [ ]:
import requests
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType,
                               LongType, ArrayType)

try:
    import notebookutils
    WORKSPACE_ID = notebookutils.runtime.context["currentWorkspaceId"]
except Exception:
    WORKSPACE_ID = spark.conf.get("trident.workspace.id")

ONELAKE = "onelake.dfs.fabric.microsoft.com"
SILVER_SCHEMA = "dbo"

def resolve_item_id(display_name: str, item_type: str) -> str:
    tok = notebookutils.credentials.getToken("pbi")
    url = f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}/items?type={item_type}"
    r = requests.get(url, headers={"Authorization": f"Bearer {tok}"}, timeout=60)
    r.raise_for_status()
    for it in r.json()["value"]:
        if it["displayName"] == display_name:
            return it["id"]
    raise ValueError(f"{item_type} '{display_name}' not found in workspace {WORKSPACE_ID}")

def item_path(item_id: str, *segments: str) -> str:
    base = f"abfss://{WORKSPACE_ID}@{ONELAKE}/{item_id}"
    return base + ("/" + "/".join(segments) if segments else "")

GE_MIRROR_ID = resolve_item_id("Graduate Exams Mirror", "MirroredDatabase")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

GE_SRC = item_path(GE_MIRROR_ID, "Tables", "university", "gradExams")
print(f"Workspace   : {WORKSPACE_ID}")
print(f"Grad mirror : {GE_MIRROR_ID}")
print(f"Source      : {GE_SRC}")

In [ ]:
raw = spark.read.format("delta").load(GE_SRC)
print("Bronze gradExams schema:")
raw.printSchema()
print(f"Bronze gradExams rows: {raw.count()}")

In [ ]:
# Normalize `undergradContext` to a struct column `uc`, whether the mirror stored it as a
# native struct or as a JSON string. This keeps the notebook robust to the mirror's representation.
uc_schema = StructType([
    StructField("overallGpa", DoubleType()),
    StructField("concentration", StringType()),
    StructField("relevantSubjects", ArrayType(StringType())),
    StructField("relevantCourseCount", LongType()),
    StructField("relevantGpa", DoubleType()),
    StructField("relevantCourses", ArrayType(StructType([
        StructField("courseNumber", StringType()),
        StructField("courseName", StringType()),
        StructField("letterGrade", StringType()),
        StructField("qualityPoints", DoubleType()),
    ]))),
])

uc_dtype = dict(raw.dtypes).get("undergradContext", "")
if uc_dtype.startswith("struct"):
    df = raw.withColumn("uc", F.col("undergradContext"))
else:
    df = raw.withColumn("uc", F.from_json(F.col("undergradContext").cast("string"), uc_schema))

In [ ]:
# ---- Fact 1: grad_exam_result (one row per exam attempt) ----
result = (df.select(
    F.col("studentId").cast("long").alias("student_id"),
    F.trim(F.col("studentName")).alias("student_name"),
    F.trim(F.col("exam")).alias("exam"),
    F.trim(F.col("targetDegree")).alias("target_degree"),
    F.col("attemptNumber").cast("int").alias("attempt_number"),
    F.to_date(F.col("testDate")).alias("test_date"),
    F.col("score").cast("int").alias("score"),
    F.col("scoreScaleMin").cast("int").alias("score_scale_min"),
    F.col("scoreScaleMax").cast("int").alias("score_scale_max"),
    F.col("admitThreshold").cast("int").alias("admit_threshold"),
    F.col("percentile").cast("int").alias("percentile"),
    F.col("passed").cast("boolean").alias("passed"),
    F.col("uc.overallGpa").cast("double").alias("overall_gpa"),
    F.trim(F.col("uc.concentration")).alias("concentration"),
    F.col("uc.relevantCourseCount").cast("int").alias("relevant_course_count"),
    F.col("uc.relevantGpa").cast("double").alias("relevant_gpa"),
).where(F.col("student_id").isNotNull())
 .dropDuplicates(["student_id", "exam", "attempt_number"]))

(result.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{SILVER_SCHEMA}.grad_exam_result"))
print(f"  dbo.grad_exam_result           rows={spark.table(f'{SILVER_SCHEMA}.grad_exam_result').count()}")

In [ ]:
# ---- Fact 2: grad_exam_relevant_course (exploded relevantCourses[]) ----
rc = (df.select(
        F.col("studentId").cast("long").alias("student_id"),
        F.trim(F.col("exam")).alias("exam"),
        F.col("attemptNumber").cast("int").alias("attempt_number"),
        F.explode_outer(F.col("uc.relevantCourses")).alias("c"),
    )
    .where(F.col("c").isNotNull())
    .select(
        "student_id", "exam", "attempt_number",
        F.trim(F.col("c.courseNumber")).alias("course_number"),
        F.trim(F.col("c.courseName")).alias("course_name"),
        F.trim(F.col("c.letterGrade")).alias("letter_grade"),
        F.col("c.qualityPoints").cast("double").alias("quality_points"),
    ))

(rc.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable(f"{SILVER_SCHEMA}.grad_exam_relevant_course"))
print(f"  dbo.grad_exam_relevant_course  rows={spark.table(f'{SILVER_SCHEMA}.grad_exam_relevant_course').count()}")

In [ ]:
# The validated correlation: first-attempt pass rate vs relevant-subject GPA.
res = spark.table(f"{SILVER_SCHEMA}.grad_exam_result")
display(
    res.where("attempt_number = 1")
       .groupBy("passed")
       .agg(F.count("*").alias("attempts"),
            F.round(F.avg("relevant_gpa"), 2).alias("avg_relevant_gpa"),
            F.round(F.avg("score"), 0).alias("avg_score"))
       .orderBy(F.col("passed").desc())
)